<div dir="rtl">

# تحويل الكتب العربية المصوّرة إلى Markdown

**قبل البدء:** ضع ملف PDF في Google Drive من هاتفك (أي مجلد).

**ثم شغّل الخلايا بالترتيب** بالضغط على زر ▶ في كل خلية:

1. **التثبيت** — عدة دقائق، مرة واحدة لكل جلسة.
2. **عرض الكتب** — يربط Drive ويعرض ملفات PDF مرقّمة.
3. **التحويل** — اكتب رقم الكتاب ثم شغّل؛ النتيجة تُحفظ في Drive.

لا نستخدم أداة رفع الملفات لأنها تتعطل على الهاتف عند انقطاع الجلسة.

⚠ **خصوصية:** المعالجة تجري على خوادم Google لا على جهازك.

</div>

In [ ]:
#@title ١) التثبيت { display-mode: "form" }
%pip install -q "paddlepaddle==3.3.1" "paddleocr[doc-parser]==3.7.0" pymupdf
!rm -rf /content/book_ocr && git clone -q https://github.com/7aidaraa/book_ocr /content/book_ocr
import sys
sys.path.insert(0, "/content/book_ocr")
print("\u2713 التثبيت اكتمل — شغّل الخلية التالية")

In [ ]:
#@title ٢) ربط Drive وعرض ملفات PDF { display-mode: "form" }
#@markdown اتركه فارغًا للبحث في Drive كله، أو اكتب اسم مجلد معيّن (مثال: `كتب`).
مجلد_البحث = ""  #@param {type:"string"}

import itertools
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

root = Path("/content/drive/MyDrive") / مجلد_البحث.strip("/ ")
if not root.exists():
    raise SystemExit(f"\u2717 المجلد غير موجود: {root}")

PDFS = sorted(itertools.islice((p for p in root.rglob("*.pdf")), 50))
if not PDFS:
    print(f"\u2717 لا توجد ملفات PDF في: {root}")
else:
    print(f"وُجد {len(PDFS)} ملف PDF:\n")
    for i, p in enumerate(PDFS, 1):
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  {i}. {p.name}  ({size_mb:.1f} م.ب)")
    print("\nاكتب رقم الكتاب في الخلية التالية ثم شغّلها.")

In [ ]:
#@title ٣) التحويل { display-mode: "form" }
رقم_الكتاب = 1  #@param {type:"integer"}
#@markdown حدّد صفحات للتجربة السريعة (اتركها 0 لتحويل الكتاب كاملًا):
أول_صفحات_فقط = 0  #@param {type:"integer"}

import os, shutil, sys
from pathlib import Path

sys.path.insert(0, "/content/book_ocr")
os.chdir("/content/book_ocr")

if not (1 <= رقم_الكتاب <= len(PDFS)):
    raise SystemExit(f"\u2717 الرقم يجب أن يكون بين 1 و {len(PDFS)}")

source_pdf = PDFS[رقم_الكتاب - 1]
print(f"الكتاب: {source_pdf.name}\n")

# نسخة عمل محلية — ملف Drive الأصلي لا يُمس
pdf_path = Path("data/input") / source_pdf.name
pdf_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(source_pdf, pdf_path)

if أول_صفحات_فقط > 0:
    import pymupdf
    with pymupdf.open(pdf_path) as doc:
        doc.select(range(min(أول_صفحات_فقط, doc.page_count)))
        trimmed = pdf_path.with_name(f"عينة-{pdf_path.name}")
        doc.save(trimmed)
    pdf_path = trimmed
    print(f"وضع التجربة: أول {أول_صفحات_فقط} صفحة فقط\n")

from app.book import process_book
from app.engines.paddleocr_engine import PaddleOCREngine

engine = PaddleOCREngine(lang="ar")
meta = process_book(
    pdf_path, engine,
    on_progress=lambda page, total, msg: print(f"[{page}/{total}] {msg}"),
)

# حفظ النتيجة في Drive ليقرأها الهاتف مباشرة
book_dir = Path("data/output") / meta["book_name"]
dest = Path("/content/drive/MyDrive/كتب-محوّلة") / meta["book_name"]
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(book_dir, dest)

failed = meta["failed_pages"]
print(f"\n\u2713 تم: {meta['page_count']} صفحة"
      + (f"، فشل منها {len(failed)}: {failed}" if failed else " — كلها نجحت"))
print(f"\nالنتيجة محفوظة في Drive:\n  كتب-محوّلة/{meta['book_name']}/book.md")
print("\n--- أول 1500 حرف من الناتج ---\n")
print((book_dir / "book.md").read_text(encoding="utf-8")[:1500])